<!-- MIGRATED_V2_TO_V3_NOTICE -->
> **ℹ️ This notebook is migrated from SageMaker Python SDK v2 to v3.**
>
> This is the v3-based version, and we recommend referring to and using this version. The SageMaker Python SDK v2 and v3 are **not backward compatible**, so v2 code will not run on a v3 installation.
>
> If you are looking for the original v2 version of this notebook, please go to the `v2-archive` branch and look for the notebook with the same name.


# Training and Deploying ML Models using JAX on SageMaker (SDK v3)


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)

---

Amazon SageMaker provides you the flexibility to train models using our pre-built machine learning containers or your own bespoke container. We'll refer to these strategies as Bring-Your-Own-Script **(BYOS)** and Bring-Your-Own-Container **(BYOC)** in this tutorial. 

### Bring Your Own JAX Script

In this notebook, we'll show how to extend our optimized TensorFlow containers to train machine learning models using the increasingly popular [JAX library](https://github.com/google/jax). We'll train a fashion MNIST classification model using vanilla JAX, another using `jax.experimental.stax`, and a final model using the [higher level Trax library](https://github.com/google/trax).

For all three patterns, we'll show how the JAX models can be serialized as standard TensorFlow [SavedModel format](https://www.tensorflow.org/guide/saved_model). This enables us to seamlessly deploy the models using the managed and optimized SageMaker TensorFlow inference containers.

### Bring Your Own JAX Container

We've included a dockerfile in this repo directory to show how you can build your own bespoke JAX container with support for GPUs on SageMaker. Unfortunately, the NVIDIA/CUDA Dockerhub containers have a [deletion policy](https://gitlab.com/nvidia/container-images/cuda/blob/master/doc/support-policy.md), so we're unable to assert that the container can be built through time. Nonetheless, you can trivially adapt a newer version of the container if your workload requires a custom container. For more information on running BYOC on SageMaker see the [documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/adapt-training-container.html).

### SDK v3 note

This notebook uses the **SageMaker Python SDK v3** APIs. The V2 framework estimator `sagemaker.tensorflow.TensorFlow` is replaced by `sagemaker.train.ModelTrainer` combined with `sagemaker.core.image_uris.retrieve` to select the managed TensorFlow container. Deployment uses the `sagemaker-core` resource classes (`Model`, `EndpointConfig`, `Endpoint`) instead of the V2 `estimator.deploy()` / `Predictor` pattern. The container-side training scripts in `training_scripts/` and their `requirements.txt` (which installs JAX) are unchanged.

In [ ]:
%pip install --upgrade 'sagemaker>=3.0'

In [ ]:
import json
from time import gmtime, strftime

from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.core import image_uris
from sagemaker.core.shapes import StoppingCondition
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
role = get_execution_role()
region = sess.boto_region_name
print(f"Region: {region}")

## Installing JAX in SageMaker TensorFlow Containers

When using BYOS with managed SageMaker containers, you can trivially install extra dependencies by providing a `requirements.txt` within the `source_dir` that contains your training scripts. At runtime these dependencies will be installed prior to executing the training script, so we can utilize our optimized TensorFlow GPU container to utilize JAX with CUDA support.

To be specific, any container that has the [sagemaker-training-toolkit](https://github.com/aws/sagemaker-training-toolkit) supports installing additional dependencies from `requirements.txt`. In the SDK v3 `ModelTrainer`, the `requirements.txt` path is passed via the `requirements` field of `SourceCode`.


## Serializing models as SavedModel format
In the upcoming training jobs we'll be training a vanilla JAX model, a Stax model, and a Trax model on the [fashion MNIST dataset](https://github.com/zalandoresearch/fashion-mnist).
The full details of the model can be seen in the `training_scripts/` directory, but it is worth calling out the methods for serialization.

The JAX/Stax models utilize the new jax2tf converter: https://github.com/google/jax/tree/master/jax/experimental/jax2tf

```python
def save_model_tf(prediction_function, params_to_save):
    tf_fun = jax2tf.convert(prediction_function, enable_xla=False)
    param_vars = tf.nest.map_structure(lambda param: tf.Variable(param), params_to_save)

    tf_graph = tf.function(
        lambda inputs: tf_fun(param_vars, inputs),
        autograph=False,
        jit_compile=False,
    )

```


The Trax model utilizes the new trax2keras functionality: https://github.com/google/trax/blob/master/trax/trax2keras.py

```python
def save_model_tf(model_to_save):
    """
    Serialize a TensorFlow graph from trained Trax Model
    :param model_to_save: Trax Model
    """
    keras_layer = trax.AsKeras(model_to_save, batch_size=1)
    inputs = tf.keras.Input(shape=(28, 28, 1))
    hidden = keras_layer(inputs)

    keras_model = tf.keras.Model(inputs=inputs, outputs=hidden)
    keras_model.save("/opt/ml/model/1", save_format="tf")
```

## Retrieve the managed TensorFlow training and inference images

In V2 the `TensorFlow` estimator resolved the container image automatically from `framework_version` / `py_version`. In V3 we retrieve the image URI explicitly with `image_uris.retrieve` and pass it to `ModelTrainer`. We retrieve both the training image (used to run the JAX scripts) and the inference image (managed TF Serving, used to host the SavedModel).

In [ ]:
TF_FRAMEWORK_VERSION = "2.10"
TF_PY_VERSION = "py39"
TRAIN_INSTANCE_TYPE = "ml.p3.2xlarge"
INFERENCE_INSTANCE_TYPE = "ml.m4.xlarge"

training_image = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version=TF_FRAMEWORK_VERSION,
    py_version=TF_PY_VERSION,
    instance_type=TRAIN_INSTANCE_TYPE,
    image_scope="training",
)

inference_image = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version=TF_FRAMEWORK_VERSION,
    py_version=TF_PY_VERSION,
    instance_type=INFERENCE_INSTANCE_TYPE,
    image_scope="inference",
)

print("Training image:", training_image)
print("Inference image:", inference_image)

## Train using Vanilla JAX

Note: Our `source_dir` directory contains a `requirements.txt` that will install JAX with CUDA support. We pass it via `SourceCode(requirements=...)`.

In [ ]:
vanilla_jax_trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    base_job_name="jax",
    source_code=SourceCode(
        source_dir="training_scripts",
        entry_script="train_jax.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type=TRAIN_INSTANCE_TYPE,
        instance_count=1,
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    hyperparameters={"num_epochs": 3},
    sagemaker_session=sess,
)
vanilla_jax_trainer.train(logs=False)

## Train Using JAX Medium-level API Stax

In [ ]:
stax_trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    base_job_name="stax",
    source_code=SourceCode(
        source_dir="training_scripts",
        entry_script="train_stax.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type=TRAIN_INSTANCE_TYPE,
        instance_count=1,
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    hyperparameters={"num_epochs": 3},
    sagemaker_session=sess,
)
stax_trainer.train(logs=False)

## Train Using JAX High-level API Trax

In [ ]:
trax_trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    base_job_name="trax",
    source_code=SourceCode(
        source_dir="training_scripts",
        entry_script="train_trax.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type=TRAIN_INSTANCE_TYPE,
        instance_count=1,
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    hyperparameters={"train_steps": 1000},
    sagemaker_session=sess,
)
trax_trainer.train(logs=False)

## Deploy Models to managed TF Containers

Since we've serialized the models as TensorFlow SavedModel format, we can host them on the managed TensorFlow Serving inference container. In V3 we do this with the `sagemaker-core` resource classes: we fetch the trained model artifact from each training job, create a `Model`, an `EndpointConfig`, and an `Endpoint`.

We define a small helper that takes a completed `ModelTrainer` and deploys its output as a real-time endpoint.

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant


def deploy_trainer(trainer, name_prefix):
    """Deploy a completed ModelTrainer's SavedModel output to a managed TF Serving endpoint."""
    suffix = strftime("%Y-%m-%d-%H-%M-%S", gmtime())
    model_name = f"{name_prefix}-{suffix}-model"
    endpoint_config_name = f"{name_prefix}-{suffix}-config"
    endpoint_name = f"{name_prefix}-{suffix}"

    # Fetch the trained model artifacts from the training job.
    model_data = trainer._latest_training_job.model_artifacts.s3_model_artifacts

    Model.create(
        model_name=model_name,
        primary_container=ContainerDefinition(
            image=inference_image,
            model_data_url=model_data,
        ),
        execution_role_arn=role,
    )

    EndpointConfig.create(
        endpoint_config_name=endpoint_config_name,
        production_variants=[
            ProductionVariant(
                variant_name="AllTraffic",
                model_name=model_name,
                instance_type=INFERENCE_INSTANCE_TYPE,
                initial_instance_count=1,
                initial_variant_weight=1,
            )
        ],
    )

    endpoint = Endpoint.create(
        endpoint_name=endpoint_name,
        endpoint_config_name=endpoint_config_name,
    )
    endpoint.wait_for_status("InService")
    return endpoint, model_name, endpoint_config_name

In [ ]:
vanilla_jax_endpoint, vanilla_jax_model_name, vanilla_jax_config_name = deploy_trainer(
    vanilla_jax_trainer, "jax"
)

In [ ]:
trax_endpoint, trax_model_name, trax_config_name = deploy_trainer(trax_trainer, "trax")

In [ ]:
stax_endpoint, stax_model_name, stax_config_name = deploy_trainer(stax_trainer, "stax")

## Test Inference Endpoints
This requires TF to be installed on your notebook's kernel as it is used to load testing data.

In V3 we invoke the endpoint with `Endpoint.invoke`, sending the TF Serving JSON payload (`{"instances": ...}`) and parsing the `predictions` field from the response.

In [ ]:
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

In [ ]:
def test_image(endpoint, test_images, test_labels, image_number):
    np_img = np.expand_dims(np.expand_dims(test_images[image_number], axis=-1), axis=0).astype(
        "float32"
    )

    payload = {"instances": np_img.tolist()}
    response = endpoint.invoke(
        body=json.dumps(payload).encode("utf-8"),
        content_type="application/json",
        accept="application/json",
    ).body.read()
    result = json.loads(response)
    pred_y = np.argmax(result["predictions"])

    print("True Label:", test_labels[image_number])
    print("Predicted Label:", pred_y)
    plt.imshow(test_images[image_number])

In [ ]:
test_image(vanilla_jax_endpoint, x_test, y_test, 0)

In [ ]:
test_image(stax_endpoint, x_test, y_test, 0)

In [ ]:
test_image(trax_endpoint, x_test, y_test, 0)

## Optional: Delete the running endpoints

In [ ]:
# Clean-Up: delete endpoints, endpoint configs, and models
for endpoint, config_name, model_name in [
    (vanilla_jax_endpoint, vanilla_jax_config_name, vanilla_jax_model_name),
    (stax_endpoint, stax_config_name, stax_model_name),
    (trax_endpoint, trax_config_name, trax_model_name),
]:
    endpoint.delete()
    EndpointConfig.get(config_name).delete()
    Model.get(model_name).delete()

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-jax_bring_your_own|sm-jax_bring_your_own.ipynb)
